## Split one 96-well plate image

This cell divides one full plate image into an **8-row × 12-column grid**, producing **96 individual well images**. The wells are numbered from left to right across each row, starting at the top-left well. Each well is saved as a separate PNG file in the selected output folder.

In [ ]:
# Cell 1: Split one image into a grid
from pathlib import Path

import numpy as np
import skimage.io

image_path = Path(r"D:\aaa_murphy_lab\BiofilmsForMachineLearning\CCM_PSH_Images\biofilm\zdoneSpliting\LitiColl_1A-3_7d_LD101.tif")
output_directory = Path(r"D:\aaa_murphy_lab\BiofilmsForMachineLearning\CCM_PSH_Images\biofilm\LD_1A-3_test")
rows = 8
columns = 12

image = skimage.io.imread(image_path)
if image.shape[0] < rows or image.shape[1] < columns:
    raise ValueError("The grid has more rows or columns than the image has pixels.")

output_directory.mkdir(parents=True, exist_ok=True)
row_edges = np.linspace(0, image.shape[0], rows + 1, dtype=int)
column_edges = np.linspace(0, image.shape[1], columns + 1, dtype=int)

segment_number = 1
for row in range(rows):
    for column in range(columns):
        segment = image[
            row_edges[row]:row_edges[row + 1],
            column_edges[column]:column_edges[column + 1],
        ]
        output_path = output_directory / f"{image_path.stem}_segment_{segment_number}.png"
        skimage.io.imsave(output_path, segment, check_contrast=False)
        segment_number += 1

print(f"Saved {rows * columns} segments to {output_directory}")


## Split multiple 96-well plate images

This cell finds every `.tif` and `.tiff` plate image in the selected input folder. It divides each image into an **8-row × 12-column grid**, producing **96 individual well images per plate**. A separate output subfolder is created for each original plate image so its wells remain grouped together.

In [ ]:
# Cell 2: Split every TIFF image in a folder into a grid
from pathlib import Path

import numpy as np
import skimage.io

input_directory = Path(r"D:\aaa_murphy_lab\Armaan_images\PSH")
output_directory = input_directory / "splitted"
rows = 8
columns = 12
image_extensions = {".tif", ".tiff"}

image_paths = sorted(
    path for path in input_directory.iterdir()
    if path.is_file() and path.suffix.lower() in image_extensions
)

if not image_paths:
    raise FileNotFoundError(f"No TIFF images found in {input_directory}")

for image_path in image_paths:
    image = skimage.io.imread(image_path)
    if image.shape[0] < rows or image.shape[1] < columns:
        print(f"Skipped {image_path.name}: image is smaller than the requested grid.")
        continue

    image_output_directory = output_directory / image_path.stem
    image_output_directory.mkdir(parents=True, exist_ok=True)
    row_edges = np.linspace(0, image.shape[0], rows + 1, dtype=int)
    column_edges = np.linspace(0, image.shape[1], columns + 1, dtype=int)

    segment_number = 1
    for row in range(rows):
        for column in range(columns):
            segment = image[
                row_edges[row]:row_edges[row + 1],
                column_edges[column]:column_edges[column + 1],
            ]
            output_path = image_output_directory / f"{image_path.stem}_segment_{segment_number}.png"
            skimage.io.imsave(output_path, segment, check_contrast=False)
            segment_number += 1

    print(f"Saved {rows * columns} segments from {image_path.name}")

print(f"Finished splitting {len(image_paths)} image(s) into {output_directory}")
